# 글자 BiLSTM 단어 분할 — 원문제 모범답안 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 GAITE **Combinatorial Word Segmentation** 모범답안과 동일): 붙어 있는 글자열을 **각 글자마다
0/1(1=단어 끝)** 로 예측. **이 노트북의 코드는 원문제 *모범답안(Ref Result)* 골격을 그대로** 따른다(암기 매핑이 1:1):

| 원문제 모범답안 | 이 노트북 |
|---|---|
| `data = list(json.items())` (word, labels) | 트윗 공백제거 → (글자열, 경계라벨) |
| `chars` → `char2idx` (0=padding) | 동일 |
| `class CompoundDataset(Dataset)` (`encode`) | 동일 (그대로) |
| `collate_fn` (패딩 + lengths) | 동일 (그대로) |
| `class MyModel` = **one-hot → BiLSTM(2층) → fc → sigmoid** | 동일 (그대로) |
| `train()` (BCELoss + 길이 마스크) | 동일 (그대로) |
| `predict_and_save` (임계값 0.6 → JSON) | 동일 (그대로) + 단어별 F1·복원 |

**데이터**: [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 트윗(공백 제거해 분할 과제로 구성).
원문제는 독일어 복합어이고 산출물도 캐글 CSV 가 아니라 `submissionval.json` 이라, 여기서도 같은 JSON 으로 저장한다.

> ⚙️ GPU 권장(작은 BiLSTM, T4 ~1분).  ⚠️ 노트북에 API 토큰이 평문 — 외부 공유 금지.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 트윗 데이터 다운로드

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_file("nlp-getting-started", "train.csv", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료")

## 2. data · char2idx (원문제 모범답안 그대로)
원문제: `data = list(json.load(f).items())` 로 (word, labels) 리스트를 얻는다. 여기선 트윗의 알파벳 단어들을
소문자로 이어 붙여 글자열을 만들고 각 글자에 `1=단어 끝` 라벨을 단다 → 같은 모양. 학습/검증으로 나눈다.

In [ ]:
import re, json, pandas as pd
df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))

def make(text):
    word, labels = "", []
    for w in re.findall(r"[a-zA-Z]+", str(text).lower()):
        word += w; labels += [0]*(len(w)-1) + [1]        # 각 단어의 마지막 글자만 1
    return word[:80], labels[:80]

alld = [make(t) for t in df["text"]]
alld = [(w, l) for w, l in alld if 4 <= len(w) <= 80 and sum(l) >= 2]
random.shuffle(alld)
cut = int(len(alld) * 0.9)
data, val_data = alld[:cut], alld[cut:]      # data = 학습용 (원문제의 train.json 역할)

# Character vocabulary (원문제 그대로: 0 은 padding 예약)
chars = sorted(list(set("".join([word for word, _ in data]))))
char2idx = {char: idx + 1 for idx, char in enumerate(chars)}
idx2char = {idx: char for char, idx in char2idx.items()}
vocab_size = len(chars)

json.dump({w: l for w, l in val_data}, open(os.path.join(WORK_DIR, "val.json"), "w"))   # predict_and_save 입력용
print("data:", len(data), "| val:", len(val_data), "| vocab_size:", vocab_size)
print("예시:", data[0][0], "|", data[0][1])

## 3. CompoundDataset & collate_fn (원문제 모범답안 그대로)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CompoundDataset(Dataset):
    def __init__(self, data, char2idx):
        self.data = data
        self.char2idx = char2idx
    def __len__(self):
        return len(self.data)
    def encode(self, word, labels):
        return (
            torch.tensor([self.char2idx[char] for char in word], dtype=torch.long),
            torch.tensor(labels, dtype=torch.float),
        )
    def __getitem__(self, idx):
        word, labels = self.data[idx]
        return self.encode(word, labels)

def collate_fn(batch):
    inputs, targets = zip(*batch)
    lengths = [len(seq) for seq in inputs]
    max_len = max(lengths)
    padded_inputs = torch.zeros(len(inputs), max_len, dtype=torch.long)
    padded_targets = torch.zeros(len(targets), max_len, dtype=torch.float)
    for i, (seq, tgt) in enumerate(zip(inputs, targets)):
        padded_inputs[i, : len(seq)] = seq
        padded_targets[i, : len(tgt)] = tgt
    return padded_inputs, padded_targets, lengths

## 4. MyModel — one-hot → BiLSTM (원문제 모범답안의 핵심: MLP 를 BiLSTM 으로)
원문제 baseline 의 `MyModel` 은 one-hot+MLP(점수 ≈0). **모범답안은 여기를 BiLSTM 으로 바꾼다**(0.95). 그대로 옮긴다.

In [ ]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, num_layers=2):
        super(MyModel, self).__init__()
        self.vocab_size = vocab_size
        self.lstm = nn.LSTM(
            vocab_size + 1,            # input_size = vocab_size + 1 (one-hot)
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)   # BiLSTM 출력 concat
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        # x: (batch, seq) → one-hot → BiLSTM → fc → sigmoid
        x = nn.functional.one_hot(x, num_classes=self.vocab_size + 1).float()
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out).squeeze(-1)   # (batch, seq)
        return self.sigmoid(logits)

## 5. train() — BCELoss + 길이 마스크 (원문제 모범답안 그대로)

In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train():
    dataset = CompoundDataset(data, char2idx)
    dataloader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
    model = MyModel(vocab_size).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    num_epochs = 16                              # 원문제 모범답안은 32
    for epoch in range(num_epochs):
        model.train(); epoch_loss = 0
        for inputs, targets, lengths in dataloader:
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs.to(device))
            # 패딩 위치 마스킹
            mask = torch.arange(inputs.shape[1])[None, :] < torch.tensor(lengths)[:, None]
            mask = mask.to(device)
            loss = criterion(outputs[mask], targets[mask])
            loss.backward(); optimizer.step(); epoch_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(dataloader):.4f}")
    return model

model = train()

## 6. predict_and_save (원문제 모범답안 그대로) → submissionval.json
글자별 출력에 임계값(0.6)을 적용해 0/1 경계를 만들어 JSON 으로 저장한다.

In [ ]:
def predict_and_save(model, input_file, output_file, char2idx, device="cpu"):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    predictions = {}
    model.eval()
    with torch.no_grad():
        for word, _ in data.items():
            indices = [char2idx.get(char, 0) for char in word]
            input_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
            outputs = model(input_tensor)[0].cpu().numpy()
            boundaries = (outputs > 0.6).astype(int)
            predictions[word] = boundaries.tolist()
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=4)
    print(f"Predictions saved to {output_file}")

predict_and_save(model, os.path.join(WORK_DIR, "val.json"),
                 os.path.join(WORK_DIR, "submissionval.json"), char2idx, device)

## 7. 단어별 F1 & 글자열 복원 확인 (라벨이 있으니 채점해 본다)

In [ ]:
from sklearn.metrics import f1_score
pred = json.load(open(os.path.join(WORK_DIR, "submissionval.json")))
true = dict(val_data)
f1s = [f1_score(true[w], pred[w], zero_division=1) for w in pred]
print(f"val 단어별 F1 평균 = {np.mean(f1s):.4f}\n")
for w, _ in val_data[:6]:
    seg = "".join(c + (" " if b else "") for c, b in zip(w, pred[w])).strip()
    print(f"{w:40s} → {seg}")
try:
    from google.colab import files; files.download(os.path.join(WORK_DIR, "submissionval.json"))
except Exception:
    pass

## 🏁 정리
원문제 **모범답안** 골격(`CompoundDataset`→`collate_fn`→**`MyModel`(one-hot BiLSTM)**→`train()`(BCELoss+마스크)
→`predict_and_save`)을 **그대로** 옮겨 글자 단위 단어 분할을 연습했다. 점수의 핵심은 *MyModel 의 MLP 를 BiLSTM 으로*
바꾸는 것. 더: num_epochs↑(원문제 32), 임계값 조정, LSTM 층/은닉차원↑, CRF.